In [2]:
def filter(df):
    sos = signal.butter(N=3, Wn=30, btype='lowpass', fs=100, output='sos')
    df['Ax'] = signal.sosfilt(sos, df['Ax'])
    df['Ay'] = signal.sosfilt(sos, df['Ay'])
    df['Az'] = signal.sosfilt(sos, df['Az'])
    return df
    

In [2]:
def create_windows(df, time_steps, step):
    windows = []
    for i in range(0, len(df)-time_steps, step):
        axs = df['Ax'].values[i: i + time_steps]
        ays = df['Ay'].values[i: i + time_steps]
        azs = df['Az'].values[i: i + time_steps]
        # Retrieve the most often used label in this segment
        windows.append([axs, ays, azs])

    # Bring the segments into a better shape
    reshaped_windows = np.asarray(windows, dtype= np.float32).reshape(-1, time_steps, N_FEATURES)
    return reshaped_windows

In [49]:
def getStatisticalFeatures(data,name):
    columns=['mean'+name,'std'+name,'median'+name,'cov'+name,'twf_perc'+name,'svf_perc'+name,'maxi'+name,'mini'+name,'skewness'+name,'kurt'+name]
    res=pd.DataFrame(columns=columns)
    for i, window in enumerate(data):
        maxi=np.amax(window)
        mini=np.amin(window)
        mean=np.mean(window, axis=None)
        std=np.std(window, axis=None)
        median=np.median(window, axis=None)
        cov=math.sqrt(std)
        twf_perc=np.percentile(window, 25, axis=None)
        svf_perc=np.percentile(window, 75, axis=None)
        skewness=skew(window)
        kurt=kurtosis(window)
        res=res.append({'mean'+name:mean,
                    'std'+name:std,
                    'median'+name:median,
                    'cov'+name:cov,
                    'twf_perc'+name:twf_perc,
                    'svf_perc'+name:svf_perc,
                    'maxi'+name:maxi,
                    'mini'+name:mini,
                    'skewness'+name:skewness,
                    'kurt'+name:kurt},ignore_index=True)
        
    return res


In [50]:
def reshape_input(x,shape): #??????
    result=x.reshape(x.shape[0],shape)
    return result

In [5]:
def featureExtraction(df):
    dataWindowed=create_windows(df, TIME_PERIODS,STEP_DISTANCE)
    #x_train=shuffle(np.array(x_train),np.array(y_train))
    statistics=[]
    statistics.append(getStatisticalFeatures(dataWindowed[::1],'_Ax'))
    statistics.append(getStatisticalFeatures(dataWindowed[::2],'_AY'))
    statistics.append(getStatisticalFeatures(dataWindowed[::3],'_Az'))
    result=statistics[0]
    result=statistics[0].merge(statistics[1],right_index=True,left_index=True)
    result=result.merge(statistics[2],right_index=True,left_index=True)
    return result